# Answering EDA (read-only)

This notebook inspects persisted Feature 10 answer traces and their immutable evidence packets. It never writes SQLite, indexes, `Courses/`, prompts, API keys, or model output. Run the normal pipeline first (for example `uv run -m uni_rag_agent ask ...`). Empty tables are expected on a fresh database.

In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

root = Path.cwd().resolve()
if root.name.lower() == "notebooks":
    root = root.parent
sqlite_path = root / "data" / "uni_rag.sqlite"
uri = sqlite_path.as_uri() + "?mode=ro"
connection = sqlite3.connect(uri, uri=True)
connection.execute("PRAGMA query_only = ON")
assert connection.execute("PRAGMA query_only").fetchone()[0] == 1

In [ ]:
def read_json(value, default):
    if value in (None, ""):
        return default
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return default
    return parsed


answers = pd.read_sql_query(
    """SELECT answers.id AS answer_id, answers.evidence_packet_id,
              answers.answer_text, answers.citations_json,
              answers.limitations_json, answers.model_name, answers.created_at,
              evidence_packets.search_run_id, evidence_packets.evidence_count
       FROM answers JOIN evidence_packets
         ON evidence_packets.id = answers.evidence_packet_id
       ORDER BY answers.id""",
    connection,
)
answers["citations"] = answers["citations_json"].map(lambda value: read_json(value, []))
answers["limitations"] = answers["limitations_json"].map(
    lambda value: read_json(value, [])
)
answers["citation_count"] = answers["citations"].map(len)
answers["limitation_count"] = answers["limitations"].map(len)
answers.head()

In [ ]:
packets = pd.read_sql_query(
    """SELECT id AS evidence_packet_id, search_run_id, evidence_count, packet_json, created_at
       FROM evidence_packets ORDER BY id""",
    connection,
)
packets["packet"] = packets["packet_json"].map(lambda value: read_json(value, {}))
packets["weakness_count"] = packets["packet"].map(
    lambda value: len(value.get("weaknesses", [])) if isinstance(value, dict) else 0
)
packets[
    ["evidence_packet_id", "search_run_id", "evidence_count", "weakness_count"]
].head()

## Checks

Citation validity is checked by resolving each `citation_id` to the packet's 1-based evidence position (with chunk-id compatibility aliases). Limitations are expected for weak packets and deterministic insufficient-evidence/safe-refusal answers. Provider/model traces are stored only as `provider:model`; prompts, keys, and invalid raw responses are intentionally absent. Injected test chat models are used by pytest and are not invoked by this notebook.

In [ ]:
def citation_validity(row):
    packet = row.get("packet", {})
    evidence = packet.get("evidence", []) if isinstance(packet, dict) else []
    by_id = {f"E{i}": item for i, item in enumerate(evidence, start=1)}
    citations = row.get("citations", [])
    return all(
        isinstance(item, dict) and (item.get("citation_id") in by_id)
        for item in citations
    )


if not answers.empty:
    joined = answers.merge(
        packets[["evidence_packet_id", "packet"]], on="evidence_packet_id", how="left"
    )
    joined["citations_valid"] = joined.apply(citation_validity, axis=1)
    joined[
        [
            "answer_id",
            "evidence_packet_id",
            "citation_count",
            "limitation_count",
            "citations_valid",
            "model_name",
        ]
    ]
else:
    joined = pd.DataFrame(
        columns=["answer_id", "evidence_packet_id", "citations_valid"]
    )
joined

In [ ]:
# Close the read-only handle when exploration is complete; no commit/write is ever issued.
connection.close()